# Welcome to the LETKF tutorial
In this tutorial we will explore the Local Ensemble Transform Kalman Filter (LETKF) using the quasi-geostrophic model. The LETKF is an ensemble-based data assimilation method that estimates background error covariances from an ensemble of model forecasts. This provides flow-dependent error estimates, unlike the static background error covariance matrix used in 3D-Var.

## Prerequisites

<u>This tutorial requires completion of the **0.qg_tutorial_start** tutorial</u>. We will use the truth run generated from that tutorial. 

It is also strongly recommended (but not required) that you complete and understand  **1.qg_3Dvar_tutorial**, since you may optionally  compare LETKF results with 3D-Var to understand the differences between static and flow-dependent background error covariances.

## Introduction to LETKF
Data assimilation aims to estimate the true atmospheric state by optimally combining a numerical model forecast (the background) with observations.  In the 3D-Var tutorial, background error covariance matrix $\mathbf{B}$ determines how observation information spreads spatially. In 3D-Var, $\mathbf{B}$ is prescribed using statistical models with fixed horizontal and vertical length scales. The same $\mathbf{B}$ matrix is used regardless of the atmospheric situation.

While highly effective for many applications, the use of a **static, climatological covariance model** imposes an important limitation: <u>the assumed error structure does not respond to the evolving flow</u>. Atmospheric forecast errors, however, are strongly flow-dependent. Errors tend to grow preferentially in dynamically-evolving regions - for example, along fronts, within jets, near convective systems, and in regions of strong baroclinicity, while remaining comparatively small in more quiescent regimes. A static covariance model cannot adapt to these changing error characteristics, potentially limiting the effectiveness of the assimilation in rapidly evolving or highly anisotropic flows.

**Ensemble Kalman filter (EnKF)** methods were developed to address this limitation by explicitly representing background uncertainty through **an ensemble of model states/forecasts**. Given an ensemble of background states, we compute the sample covariance from the ensemble perturbations. 

<center>$\mathbf{B} \approx \mathbf{P}_b = \frac{1}{N-1} \mathbf{X}_b \mathbf{X}_b^T,$</center>

where $\mathbf{X}_b$ contains the ensemble perturbations about the ensemble mean, using ${N}$ ensemble members. Because the ensemble members evolve through the nonlinear model, the resulting covariance estimate is **flow-dependent** and captures the errors of the day.

Among the various EnKF formulations, the **Local Ensemble Transform Kalman Filter (LETKF)** has emerged as a widely used and computationally efficient approach, particularly for high-dimensional geophysical systems. The LETKF performs the analysis independently at each model grid point by assimilating only nearby observations, transforming the ensemble into a reduced-dimensional subspace spanned by the ensemble perturbations. More specific information about the LETKF can be found in [Hunt et al. 2007](https://www.sciencedirect.com/science/article/abs/pii/S0167278906004647)

#### Issues and Mitigation Strategies
Despite the appeal of estimating flow-dependent errors from an ensemble, the practical application of EnKF methods involves several important considerations. Operational ensembles contain perhaps 20 to 100 members, while the atmospheric state dimension may exceed *one billion*. Estimating a covariance matrix from such limited samples introduces **sampling error**, manifesting as spurious long-range correlations that may contaminate the analysis update. 

While increasing ensemble size can reduce the sampling error issue, it is impractical for most state-of-the-art NWP applications. However another method, **covariance localization**,  addresses spurious correlations by tapering the covariance to zero beyond some distance, effectively using only nearby ensemble members for each grid point, suppressing unphysical correlations and ensuring that observational information is applied at appropriate spatial scales. 

At the same time, finite ensembles tend to severely underestimate forecast uncertainty after the EnKF analysis update of the ensemble, necessitating the use of **covariance inflation** to maintain ensemble spread and prevent filter divergence.

These design choices -  <u>ensemble size, localization strategy, and inflation method</u> — strongly influence the performance of the LETKF and other EnKF-based methods, and must be carefully tuned for a given model and observing system.

## Environment Setup




The following is a quick overview of where the files are. To do this, you can navigate the menu on your left or you can list the files under the different directories.<br><br>
Now for using Jupyter Labs notebooks: you can click on any box that contains code (like the one below) and hit play, it will run the piece of code there!
<br>
<center> <img src="images/run_command.png"/> </center>
<br>
<br>

We now set environment variables and verify that required executables exist. **You need to do this anytime you open this notebook for a new localhost session**

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
export LETKF_BASE=$JEDI_EDU/qgLETKF

In [ ]:
# Verify LETKF executables exist
ls $JEDI_BIN/qg_gen_ens_pert_B.x
ls $JEDI_BIN/qg_letkf.x
ls $JEDI_BIN/qg_ens_variance.x


### Table of LETKF Experiments

| Experiment Name | letkf yaml file | Description |
| --- | --- | --- |
| exp_default | letkf.yaml | Default single-observation LETKF experiment
| exp_dbl_loc | letkf_dbl_loc.yaml | Double the localization scale for LETKF
| exp_5mem | letkf_5mem.yaml | Reduces ensemble size to 5
| exp_mult_obs  | letkf_mult_obs.yaml | Assimilates 100 observations


# Experiment 1: LETKF with one observation
***

### Step 1 & 2: Verify that truth and background ensemble exist

Double-check the  files exist from teh qgstart tutorial. If not, please run that tutorial first!

In [ ]:
ls $JEDI_EDU/qgstart/output/truth

In [ ]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/bg/bkgd.ens.*P1D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Background files don't exist! Please see the 0.qg_tutorial_start tutorial to generate the background ensemble"
fi

You can have a look at the different ensembles produced by comparing the errors relative to the truth. We will examine three members: 1, 10, and 20

In [ ]:
cd $JEDI_EDU/plots_scripts
for ensmem in 1 10 20; do
   python ./plot.py qg fields \
        $JEDI_EDU/qgstart/output/bg/bkgd.ens.${ensmem}.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_default/plots/bkgd_error_mem${ensmem} --title "background error ens ${ensmem}"
done

![ensmems](images/ens_mem1-10-20_x.png "Ensemble Member Errors Example")

Before generating the observations, we should also take a moment to calculate and examine the ensemble variances of the ensemble that was created in the qgstart tutorial. We will do this by running the `qg_ens_variance.x` program, with the supplied `ens_variance.yaml` file, copied here:

```yaml
background:
  date: 2009-12-31T00:00:00Z
  filename: qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.P1D.nc
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: 2009-12-31T00:00:00Z
      filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc
      state variables: [x,q,u,v]
    pattern: %mem%
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgLETKF/output/exp_default/bg
  exp: variance
  type: diag
  date: 2009-12-31T00:00:00Z
```

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_ens_variance.x ./qgLETKF/yamls/ens_variance.yaml

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_default/bg/variance.diag.2009-12-31T00\:00\:00Z.nc \
        --fieldmax 3e15 --output $JEDI_EDU/qgLETKF/output/exp_default/plots/bkgd_ens_variance --title "background ensemble variance"

![ensvar](images/ens_variance_x_u.png "Ensemble Background Variance")

The ensemble variance shows areas of enhanced uncertainty, as computed from the ensemble itself. That is, areas with higher variance have higher uncertainty, and within the EnKF we would expect these higher-variance regions to be corrected more strongly towards observations than lower-variance regions (for an observation with the same y-H(xb) innovation). 

### Step 3: Generate observation

To help highlight the impact of flow-dependence, we will produce a new observation at a different location than the qgstart. We will choose the observation location depicted in the variance plot above, nearby a high-variance region in streamfunction.

We can simply rerun the `qg_hofx.x` program from the qgstart tutorial, modifying only the observation latitude and longitude.

Take a look at ``makeobs3d_oneobs_newloc.yaml``, and compare it to the original yaml file from the qgstart tutorial ``makeobs3d_oneobs.yaml``


In [ ]:
cd $JEDI_EDU
diff ./qgstart/yamls/makeobs3d_oneobs.yaml ./qgLETKF/yamls/makeobs3d_oneobs_newloc.yaml 

**(NOTE the error code of 1 shown at the bottom after running the `diff` command does not indicate anything went wrong. It simply indicates that the command shows the files are different.)**

As you can see, we have changed the $lon$, $lat$ to $[150]$ and $[50]$, close to the northeastern part of our plot domain, the same as the dot shown in the above variance figure. We have also changed the obsfile output name and directory location for the new observation.

Next, run the hofx3d program to create this observations.

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgLETKF/yamls/makeobs3d_oneobs_newloc.yaml

We can check that the file was created by extracting its location in a text file

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgLETKF/output/exp_default/obs/truth_oneob_newloc.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_default/plots/qg_oneob_newloc

In [ ]:
cat $JEDI_EDU/qgletkf/output/exp_default/plots/qg_oneob_newloc.txt

We can now run the LETKF!

### Step 4: Run the LETKF!

We are now in a position to perform the LETKF data assimilation using the ``qg_letkf.x`` executable. Have a look at `letkf.yaml` and see if you can understand every line:

```yaml
indow begin: &date_bgn 2009-12-30T18:00:00Z
window length: PT12H

geometry:
  nx: 40
  ny: 20 
  depths: [4000.0, 6000.0]

background:
  members from template:
    template:
      states:
      - date: &date_mid 2009-12-31T00:00:00Z
        filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc
    pattern: %mem%
    nmembers: 20                       # Number of ensemble members to analyze

observations:
  observers:
  - obs operator:
      obs type: Stream
    obs space:
      obsdatain:
        engine:
          obsfile: qgLETKF/output/exp_default/obs/truth_oneob_newloc.obs3d.nc
      obsdataout:
        engine:
          obsfile: qgLETKF/output/exp_default/obs/letkf_oneob_newloc.obs3d.nc
      obs type: Stream
    obs error:
      covariance model: diagonal        # Obs errors genetally considered uncorrelated in R
    obs localizations:                  # Localization is handled in observation space              
    - localization method: Heaviside    # 'Heaviside' is a step function (1 if within region, 0 outside of region)
      lengthscale: 4.0e6                # Horizontal length scale used for Heaviside function in localization
                                        # Note no vertical localization is applied 
                       
driver:
  update obs config with geometry info: false
  save prior mean: true                 # Save the ensemble mean background/prior if true
  save posterior mean: true             # Save the ensemble mean analysis/posterior if true
  save posterior mean increment: true   # Save the ensemble mean analysis increments if true

local ensemble DA: 
  solver: LETKF  # Controls which flavor of ensemble solver is used
  inflation:    # Covariance inflation (applied after LETKF update)
    rtpp: 0.5   # Relaxation to prior perturbations
    mult: 1.1   # Multiplicative inflation factor

output: # Controls output of different ensemble member analyses
  states:  
  - datadir: qgLETKF/output/exp_default/da
    date: *date_mid
    exp: letkf.%{member}%
    type: an
  
output increment: # Controls output of ensemble mean analysis increments
  datadir: qgLETKF/output/exp_default/da
  exp: letkf_meaninc
  date: *date_mid
  type: an

output mean prior: # Controls output of ensemble mean background
  datadir: qgLETKF/output/exp_default/da
  exp: letkf_meanprior
  date: *date_mid
  type: an

```

Run this code via

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgLETKF/yamls/letkf.yaml 

There will be several results in `/home/nonroot/shared/EDU/qgLETKF/output/exp_default/da`. List them with the command below, do you know what each file represents?

In [ ]:
ls $JEDI_EDU/qgLETKF/output/exp_default/da

**One important note for the file list above. The posterior analysis ensemble mean is the "000000" ensemble member.** 

We will now plot the results using the python plotting script. Let's examine the mean increments. That is, the mean ensemble analysis minus the mean ensemble background. (Alternatively, you could plot the contents of the `da/letkf_meaninc.an.2009-12-31T00:00:00Z.nc` file directly)

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_default/da/letkf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLETKF/output/exp_default/da/letkf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgLETKF/output/exp_default/obs/letkf_oneob_newloc.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLETKF/output/exp_default/plots/letkf_mean_increments --title "ens. mean anl inc."

#### LETKF Ensemble Mean Increments
![meaninc](images/ensmean_inc_default_allvars.png "Mean Analysis Increments")

Examine the LETKF mean increments (pasted above for convenience) and try to think about answers to these questions:
- Do you notice a pattern to the LETKF increments, especially streamfunction?
- Why do you suppose the maximium increment magnitude in streamfunction is displaced away from the observations?
- *(Optional, if you have completed the 3DVar tutorial)* How do these increments compare to 3dvar increments? Use the below image of 3dvar increments previously produced for comparison.

Hint: the ensemble variance plot holds the key to some of these insights! 

Regarding the 2nd question, this is a somewhat extreme and unique scenario that is meant to highlight how the **flow-dependence** of LETKF manifests in the analysis, relative to the ensemble that it uses for error covariances. In practice, most of the time the maximum magnitude increments will be centered around the observation, in part because the most commonly-used localization functions (such as the **Gaspari-Cohn** or GC function) are weights that smoothly taper from 1 at observation location to 0 at the user-defined cutoff distance. But in this case, the **heaviside function** is used for localization (see `letkf.yaml`), which is a 'top hat'-shaped function without tapering (either 1 within the prescribed radius, or 0 outside of it). Combined with the high ensemble background variance region being slightly displaced from the observation location but still within its localization radius (set to 4000km), this explains why the maximum magnitude increment in streamfunction is offset to the northwest of the observation.

#### 3DVar Increments:
![3dvar_increments]( images/3dvar_inc_newloc.png "3DVar Analysis Increments") 


# Experiment 2: Change the horizontal localization length scale

***

In the experiment **exp_dbl_loc**, we increase the localization length scale, so instead of using the same as in **exp_default** `lengthscale 4.0e6` we will use `lengthscale: 8.0e6`. We will follow the same 4-step procedure as before.<br>


### Steps 1-3: Truth, Background Ensemble, Observations

The truth and background ensemble remain unchanged. We will copy the observation file produced from the default experiment to the **exp_dbl_loc** folder 

In [ ]:
cp $JEDI_EDU/qgLETKF/output/exp_default/obs/truth_oneob_newloc.obs3d.nc $JEDI_EDU/qgLETKF/output/exp_dbl_loc/obs
ls $JEDI_EDU/qgLETKF/output/exp_dbl_loc/obs

### Step 4: Run the data assimilation:
Let's compare `letkf_dbl_loc.yaml` to the default experiment `letkf.yaml`:

In [ ]:
diff $JEDI_EDU/qgLETKF/yamls/letkf.yaml $JEDI_EDU/qgLETKF/yamls/letkf_dbl_loc.yaml

The main parameter that changes the length scale for localization is `lengthscale`, which is doubled to 8.0e6 (i.e. 8000 km). There are also changes to the output file locations for the analysis that replaced  `exp_default` by `exp_dbl_loc`, ensuring we do not overwrite any previous results.

You may run the LETKF:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgLETKF/yamls/letkf_dbl_loc.yaml

<br>Before you look at the results, take a moment to think and form an idea of what you expect to see. After that, have a look at the results via the increment figures, generated via:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_dbl_loc/da/letkf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLETKF/output/exp_dbl_loc/da/letkf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgLETKF/output/exp_dbl_loc/obs/letkf_oneob_newloc.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLETKF/output/exp_dbl_loc/plots/letkf_mean_increments --title "ens. mean anl inc."

#### Ensemble Mean Increments - Doubled Loc
![meaninc_dbl](images/ensmean_inc_dbl_loc.png "Mean Analysis Increments - Doubled Loc")

Examining the increments, we see now much larger area where corrections are being made. Think carefully about how to interpret this
- Do you trust the increments far away from the observation are realistic? Why or why not?
> - <u>*(Optional, if you have completed the 3DVar tutorial)*</u> How does this compare to 3dvar and the lengthscale for static B?<br><br>
>Though the result of changing localization looks a lot like how we changed the horizontal_length_scale for 3dvar (Expanding/contracting region of increments when length scale is increased/decreased), it is important to understand the key differences between the two. First, in 3dvar the length scale *defines* how we model isotropic covariances for B. In contrast for LETKF, localization does not itself define B, but acts instead as a **filter over the ensemble flow-dependent estimate of B**. Though both demand properly-tuned settings for best results, they represent different concepts.

# Experiment 3: LETKF with reduced ensemble members


### Steps 1-3: Truth, Background Ensemble, Observations

The truth and background ensemble remain unchanged. We will copy the observation file produced from the default experiment to the **exp_5mem** folder 


In [ ]:
cp $JEDI_EDU/qgLETKF/output/exp_default/obs/truth_oneob_newloc.obs3d.nc $JEDI_EDU/qgLETKF/output/exp_5mem/obs
ls $JEDI_EDU/qgLETKF/output/exp_5mem/obs

### Step 4: Run LETKF
Let us compare `letkf_5mem.yaml` to `letkf_default.yaml`. 


In [ ]:
diff $JEDI_EDU/qgLETKF/yamls/letkf.yaml $JEDI_EDU/qgLETKF/yamls/letkf_5mem.yaml
#ls $JEDI_EDU/qgLETKF/yamls/letkf_5mem.yaml

You can see, we have modified `nmembers` from 20 to 5, in addition to the directory path changes to `exp_5mem`,

*You may also notice that we are using the doubled-localization setting here (`lengthscale: 8.0e6`)  as in `exp_dbl_loc`. This is to help demonstrate the impact of using fewer ensemble members more clearly in the analysis.*

Once that is done, you may run the LETKF again with `letkf_5mem.yaml`

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgLETKF/yamls/letkf_5mem.yaml 

And now plot the increments. Think carefully what you might see, and how it might compare to the LETKF run using 20 members

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_5mem/da/letkf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLETKF/output/exp_5mem/da/letkf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgLETKF/output/exp_5mem/obs/letkf_oneob_newloc.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLETKF/output/exp_5mem/plots/letkf_mean_increments --title "ens. mean anl inc."

![meaninc_5mem_x](images/ensmean_inc_5mem_x.jpg "Mean Analysis Increments - x")

**To help analyze the difference among these experiments with varying ensemble members, let's compute and plot ensemble variance**

First examine how ens_variance.yaml is modified from experiment one to reduce `nmembers` from 20 to 5 (provided already in `ens_variance_5mem.yaml`). Then, let us compute the 5-member variance by running `qg_ens_variance.x`.


In [ ]:
diff $JEDI_EDU/qgLETKF/yamls/ens_variance.yaml $JEDI_EDU/qgLETKF/yamls/ens_variance_5mem.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_ens_variance.x ./qgLETKF/yamls/ens_variance_5mem.yaml

#### Now plot each variance result and compare for using 5 and 20 members. Try to relate the variance plots to the increment plot differences. Can you see any issues with using a reduced ensemble size?

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_5mem/bg/variance.diag.2009-12-31T00\:00\:00Z.nc \
        --fieldmax 6e15 --output  $JEDI_EDU/qgLETKF/output/exp_5mem/plots/bkgd_ens_variance_nmem5 --title "bkg variance 5 members"
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_default/bg/variance.diag.2009-12-31T00\:00\:00Z.nc \
        --fieldmax 6e15 --output $JEDI_EDU/qgLETKF/output/exp_5mem/plots/bkgd_ens_variance_nmem20 --title "bkg variance 20 members"


![ensvar_5mem](images/ensvar_5vs20mem.png "Ensemble Variance - 5 vs 20 members")

<br>Study the results and try to answer the following questions:

- Do you trust the 20-member or 5-member results more?
- Is 5 members sufficient or too small to be reliable?
- How can we be sure which variances are "more correct"?

*Hint: Where are the maximum magnitude increments located for the 5-mem run? Does it make physical sense?*


# Experiment 4: Assimilate multiple observations
***

We now embark on a more realistic situation in which 100 observations are assimilated all at once.

### Steps 1-3: Truth, Background Ensemble, Observations

The truth and background ensemble remain unchanged. We will copy the 100-observations file produced from the **0.qg_tutorial_start** tutorial  to the **exp_mult_obs** folder 


In [ ]:
cp $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs
ls $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs

### Step 4: Run the LETKF

Let's compare the 100-obs settings for `letkf_mult_obs.yaml` to the single-observation settings in `letkf.yaml`

In [ ]:
diff $JEDI_EDU/qgLETKF/yamls/letkf.yaml $JEDI_EDU/qgLETKF/yamls/letkf_mult_obs.yaml

The only modification is to the input `obsfile:`, using now `truth_100obs.obs3d.nc`. We have also changed the output folder paths from `exp_default` to `exp_mult_obs`.

Next, let's run the LETKF

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_letkf.x ./qgletkf/yamls/letkf_mult_obs.yaml

<br><br> You can see the results, after thinking about your expectations, via:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_mult_obs/da/letkf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLETKF/output/exp_mult_obs/da/letkf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        --plotObsLocations $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/letkf_meaninc --title "analysis increment ENSMEAN"

![meaninc_mult](images/ensmean_inc_x_diff.jpg "Mean Increment - 100 obs")

- Do you expect the increments to be larger or smaller, or similar to the default experiment? Why?

You can also look at and compare ensemble mean background and analysis errors via:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_mult_obs/da/letkf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/letkf_meananl_error --title "mean analysis error" 
python ./plot.py qg fields \
        $JEDI_EDU/qgLETKF/output/exp_mult_obs/da/letkf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/letkf_meanbkg_error --title "mean background error" 

![meanerr_mult](images/ensmean_err_bkg_an_x_mult_obs.png "Mean Error - Background vs Analysis")
- Is the analysis error indeed smaller than the background error?
- In every grid point? Or on average?

We will perform observation-space diagnostics as we did in the 3D-var tutorial. Thus we need to extract the hofx for the background and analysis at observation locations with `qg_hofx3d.x`. For your convenience, these files have already been provided within the qgLETKF/yamls directory, so you will only need to run these programs:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgLETKF/yamls/hofx_mult_obs.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgLETKF/yamls/hofx_mult_obs_an.yaml

This results in two files in `exp_mult_obs/obs` directory:, `hofx_meananl.obs3d.nc` and `hofx_meanbkg.obs3d.nc`. We can extract the contents of these observation files by using the python script:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs/hofx_meananl.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/hofx_meananl
cat $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/hofx_meananl.txt

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgLETKF/output/exp_mult_obs/obs/hofx_meanbkg.obs3d.nc \
        --output $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/hofx_meanbkg
cat $JEDI_EDU/qgLETKF/output/exp_mult_obs/plots/hofx_meanbkg.txt

In the **3D-var** tutorial, we use these hofx diagnostics to overlay and visualize corrections from many observations to the model, relating the observation innovations and analysis fits to the background error and resulting analysis error after DA. That can also be repeated for these LETKF results, left as an exercise to the reader.

In this tutorial, we will introduce computation of statistics, namely **Root-mean-square error (RMSE).**  
RMSE is a commonly used metric for quantifying the difference between a model state and a set of observations. Given a model estimate $\mathbf{x}$ and corresponding observations $\mathbf{y}$, the RMSE is defined as

<center>$\mathrm{RMSE} = \sqrt{\frac{1}{M} \sum_{i=1}^{M} \left( \mathcal{H}_i(\mathbf{x}) - y_i \right)^2},$</center>

where $M$ is the number of observations and $\mathcal{H}_i(\mathbf{x})$ denotes the model state mapped into observation space via the observation operator $\mathcal{H}$.

Physically, RMSE measures the typical magnitude of model differences relative to observations, expressed in the same units as the observed variable. Squaring the differences emphasizes larger discrepancies, making RMSE particularly sensitive to outliers and systematic biases. As a result, RMSE reflects both random error and unresolved systematic error in the model estimate.

In DA, RMSE is often computed using innovations (background-minus-observation or analysis-minus-observation differences) to assess forecast quality and the impact of assimilation. In particular, **the background RMSE provides a measure of forecast skill prior to assimilation, while the analysis RMSE quantifies how effectively the analysis <u>fits</u> to the assimilated observations. (The latter is sometimes referred to root-mean-square-differences (RMSD) because it is not truly a measure of error.)**. When computed consistently over time and many cycles, RMSE serves as a key diagnostic for tuning assimilation parameters such as covariance inflation, localization length scales, and observation error assumptions. It also is a good indicator of the stability of the DA system, as will be explored in the cycling tutorial.

It is important to note that RMSE depends on the representativeness and error characteristics of the observations. Because observations themselves contain error, RMSE should be interpreted as a measure of model–observation mismatch, not a direct estimate of true state error unless observation errors are negligible or explicitly accounted for.

Here, the script `computermsd.bash` is provided within the `plot_scripts` directory to compute and print out this RMSE statistic for you. We need to provide it the text files produced above to run:


In [ ]:
cd $JEDI_EDU/plots_scripts
PLOT_DIR=$JEDI_EDU/qgLETKF/output/exp_mult_obs/plots
# To run: ./computermsd.bash <hofx_background_file.txt> <hofx_analysis_file.txt>
./computermsd.bash $PLOT_DIR/hofx_meanbkg.txt $PLOT_DIR/hofx_meananl.txt


- Do you understand the value that was produced? What units are they in?
- Is the anlaysis fit to observations indeed lower than the background innovations, on average?

#### (Optional: comparison to 3D-Var)
If you have run the 3D-var tutorial already, you can also compare these RMSE values to the **exp_mult_obs** experiment conducted within the 3D-var tutorial:

In [ ]:
cd $JEDI_EDU/plots_scripts
PLOT_DIR=$JEDI_EDU/qg3Dvar/output/exp_mult_obs/plots
# To run: ./computermsd.bash <hofx_background_file.txt> <hofx_analysis_file.txt>
./computermsd.bash $PLOT_DIR/qg_multobs_bkgd.txt $PLOT_DIR/qg_multobs_an.txt

- How do these values compare?
- Why is the background error different?
- Is this truly a fair comparison? If not, how would we adjust these experiments for a true comparison between solvers?

The last question is important. The short answer is although LETKF appears to more accurately fit observations than 3D-Var, we cannot definitively say this is true yet. The LETKF computed the observation innovations from the **10-member ensemble mean**, whereas the 3D-var computed it against simply the first generated background member. Thus, the backgrounds given to each are different.

The answer to how we make the comparison fairer is straightforward: **we need to ensure both experiments are using the same background state for the analysis.** Thus, if we were instead to give 3D-var the **ensemble mean background/prior** to update the anlaysis with, then we could make a fairer comparison between the two solvers. This is in fact what was provided in the single-observation 3dvar images above for comparison.

Further, to compare the true errors of each solver, we would not wnat to look at analysis fit to observations, since that does not necessarily reflect overall performance of the DA solver. Overfitting to observations can lead to instabilities and thus higher errors in the forecast as well.  Thus instead, we must asses **forecast error**. That is, if we were to take these analyses and run the forecast to the next cycle, then we can compare the RMSE values directly and haev a better idea of how to judge the performance between each.

# Summary

In this tutorial we explored the Local Ensemble Transform Kalman Filter using the quasi-geostrophic model. We investigated how ensemble size, localization, and observation density affect the LETKF analysis.

### Key Concepts

1. **Flow-dependent background error covariances:** Unlike 3D-Var's static B matrix, the LETKF estimates B from an ensemble of forecasts, providing error estimates that adapt to the current atmospheric situation.

2. **Localization:** Suppresses spurious correlations from finite ensemble sampling by tapering covariances to zero beyond a specified distance. Localization is the most sensitive LETKF parameter.

3. **Ensemble size:** Larger ensembles provide better sampling of the background error covariance, producing more accurate flow-dependent error estimates.

4. **Inflation:** Primarily important for cycling data assimilation to prevent ensemble spread collapse. 
